# Übersetzungs-Test RM ⇄ DE (Colab)

Modell: [ZurichNLP/romansh-nllb-1.3b-ct2](https://huggingface.co/ZurichNLP/romansh-nllb-1.3b-ct2) · Lizenz CC-BY-NC-4.0 (nicht kommerziell)

**Nur mit unkritischem Testtext verwenden.** Der Text wird auf einem Google-Server verarbeitet, nicht auf dem eigenen Rechner.

**Ablauf**
1. Menü *Laufzeit → Laufzeittyp ändern → T4-GPU* wählen (viel schneller; ohne GPU geht es auch, dauert aber länger).
2. *Laufzeit → Alle ausführen*. Beim ersten Mal dauert Zelle 1–2 ein paar Minuten (Installation, Modell-Download ca. 1.3 GB).
3. In Zelle 3 den Text einfügen (eine Frase pro Zeile, z. B. aus dem Studio-Test: «Copiar & avrir ALAS»), Richtung und Idiom wählen, auf *Übersetzen* klicken.
4. Ergebnis kopieren und im Studio-Test in Schritt 3 einfügen.

In [ ]:
#@title 1 · Installieren
!pip -q install ctranslate2 sentencepiece huggingface_hub transformers

In [ ]:
#@title 2 · Modell laden
import ctranslate2, time
from huggingface_hub import snapshot_download
from transformers import NllbTokenizer
t0 = time.time()
path = snapshot_download("ZurichNLP/romansh-nllb-1.3b-ct2")
tok = NllbTokenizer.from_pretrained(path)
dev = "cuda" if ctranslate2.get_cuda_device_count() > 0 else "cpu"
tr = ctranslate2.Translator(path, device=dev)
print(f"Modell bereit auf {dev.upper()} ({time.time()-t0:.0f} s)")

def translate(src, tgt, lines, bs=16):
    tok.src_lang = src
    out = []
    for i in range(0, len(lines), bs):
        part = lines[i:i+bs]
        batch = [tok.convert_ids_to_tokens(tok.encode(t)) for t in part]
        res = tr.translate_batch(batch, target_prefix=[[tgt]]*len(batch), beam_size=4, max_decoding_length=256)
        out += [tok.decode(tok.convert_tokens_to_ids(r.hypotheses[0][1:]), skip_special_tokens=True) for r in res]
    return out

In [ ]:
#@title 3 · Übersetzen
import ipywidgets as w
from IPython.display import display, Javascript

IDIOMS = {"Rumantsch Grischun":"rm-rumgr","Sursilvan":"rm-sursilv","Sutsilvan":"rm-sutsilv","Surmiran":"rm-surmiran","Puter":"rm-puter","Vallader":"rm-vallader"}
richtung = w.ToggleButtons(options=["RM → DE","DE → RM"], description="Richtung")
idiom = w.Dropdown(options=list(IDIOMS), description="Idiom")
inp = w.Textarea(placeholder="Eine Frase pro Zeile …", layout=w.Layout(width="100%", height="180px"))
btn = w.Button(description="Übersetzen", button_style="primary")
status = w.HTML()
out = w.Textarea(layout=w.Layout(width="100%", height="180px"))
cp = w.Button(description="Ergebnis kopieren")

def run(_):
    lines = [l.strip() for l in inp.value.splitlines()]
    if not any(lines): status.value = "Kein Text."; return
    rm = IDIOMS[idiom.value]
    src, tgt = (rm, "deu_Latn") if richtung.value == "RM → DE" else ("deu_Latn", rm)
    status.value = "Übersetze …"
    # leere Zeilen bleiben leer, damit die Zeilenzahl stimmt
    idx = [i for i,l in enumerate(lines) if l]
    res = translate(src, tgt, [lines[i] for i in idx])
    full = [""]*len(lines)
    for i,r in zip(idx,res): full[i] = r.replace("\n"," ").strip()
    out.value = "\n".join(full)
    status.value = f"✓ {len(idx)} Zeilen übersetzt – bitte prüfen."
def copy(_):
    display(Javascript("const t=[...document.querySelectorAll('textarea')].pop(); t.select(); document.execCommand('copy');"))
btn.on_click(run); cp.on_click(copy)
display(w.VBox([richtung, idiom, inp, btn, status, out, cp]))